- frame_dir (str): The identifier of the corresponding video. (name of file)
- total_frames (int): The number of frames in this video. (len of 'keypoints')
- img_shape (tuple[int]): The shape of a video frame, a tuple with two elements, in the format of (height, width). Only required for 2D skeletons. (got it)
- original_shape (tuple[int]): Same as img_shape. (got it)
- label (int): The action label. ('overhead press')
- keypoint (np.ndarray, with shape [M x T x V x C]): The keypoint annotation. M: number of persons; T: number of frames (same as total_frames); V: number of keypoints (25 for NTURGB+D 3D skeleton, 17 for CoCo, 18 for OpenPose, etc. ); C: number of dimensions for keypoint coordinates (C=2 for 2D keypoint)
- keypoint_score (np.ndarray, with shape [M x T x V]): The confidence score of keypoints. Only required for 2D skeletons.


In [3]:
# %pip install pandas

import pandas as pd
import json
import pickle
import os

In [4]:
# Settings
base_dir = '../../data'
# sample_class = 'correct'  # 'knees_error', 'elbows_error'
# sample_class = 'knees_error' #'correct'  'knees_error', 'elbows_error'
sample_class = 'elbows_error'  # 'knees_error', 'elbows_error'

extract_main_person = False

In [5]:
# Path to the folder with JSON files
json_folder = os.path.join(base_dir, 'ohp_poses', sample_class)

# Dictionary to store all loaded JSON data
all_data = {}

# Loop through all .json files in the folder
for filename in os.listdir(json_folder):
    if filename.endswith('.json'):
        filepath = os.path.join(json_folder, filename)
        with open(filepath, 'r') as file:
            try:
                data = json.load(file)
                key = os.path.splitext(filename)[0]  # filename without .json
                all_data[key] = data
            except json.JSONDecodeError:
                print(f"⚠️ Could not parse {filename}, skipping.")

# Example: print one loaded entry
print(all_data.keys())

dict_keys(['62805_6', '62866_3', '62868_4', '62876_2', '62947_2', '62989_7', '62992_5', '62993_1', '63009_4', '63028_3', '63041_9', '63094_1', '63097_1', '63106_6', '63159_8', '63164_2', '63203_3', '63207_1', '63241_1', '63248_1', '63296_2', '63309_8', '63313_2', '63364_1', '63378_5', '63390_7', '63422_3', '63425_1', '63427_4', '63451_3', '63474_2', '63526_1', '63672_1', '63706_7', '63725_6', '63729_5', '63766_9', '63783_4', '63784_2', '63815_1', '63886_1', '63910_2', '63915_6', '63918_2', '63919_3', '63992_1', '63995_1', '64000_1', '64071_1', '64079_2', '64099_1', '64108_11', '64120_1', '64142_1', '64164_1', '64194_1', '64198_6', '64232_1', '64271_3', '64307_1', '64336_1', '64349_7', '64361_2', '64386_3', '64420_3', '64450_2', '64460_3', '64482_1', '64503_2', '64546_1', '64591_3', '64607_2', '64624_3', '64673_3', '64702_1', '64708_1', '64710_3', '64775_6', '64798_1', '64799_1', '64800_1', '64804_3', '64893_3', '64899_1', '64908_5', '64922_5', '64935_5', '65009_1', '65010_1', '65025_1'

In [6]:
# print('No. people: ',len(all_data.get('62794_6').get('keypoints')[0].keys()))

all_data


In [7]:
# print('No. frames: ', len(all_data.get('62794_6').get('keypoints')))

In [8]:
# %pip install opencv-python

In [9]:
import cv2

# Path to the folder containing .mp4 videos
video_folder = os.path.join(base_dir, 'ohp_labeled', sample_class)

# Dictionary to hold video metadata
video_info = {}

# Loop through all files in the folder
for filename in os.listdir(video_folder):
    if filename.lower().endswith('.mp4'):
        video_path = os.path.join(video_folder, filename)
        video_name = os.path.splitext(filename)[0]

        # Open video file
        vid = cv2.VideoCapture(video_path)

        if not vid.isOpened():
            print(f"❌ Failed to open: {filename}")
            continue

        # Get properties
        width = vid.get(cv2.CAP_PROP_FRAME_WIDTH)
        height = vid.get(cv2.CAP_PROP_FRAME_HEIGHT)
        fps = vid.get(cv2.CAP_PROP_FPS)
        frame_count = vid.get(cv2.CAP_PROP_FRAME_COUNT)
        duration = frame_count / fps if fps else 0

        # Store in dictionary
        video_info[video_name] = {
            "width": int(width),
            "height": int(height),
            "fps": round(fps, 2),
            "frame_count": int(frame_count),
            "duration_sec": round(duration, 2)
        }

        vid.release()

# Print or save the results
output_path = os.path.join(video_folder, 'video_properties.json')
with open(output_path, 'w') as f:
    json.dump(video_info, f, indent=2)

print(f"✅ Processed {len(video_info)} videos. Info saved to: {output_path}")

✅ Processed 459 videos. Info saved to: ../../data\ohp_labeled\elbows_error\video_properties.json


In [10]:
with open(os.path.join(video_folder, 'video_properties.json'), 'r') as file:
    video_properties = json.load(file)

In [11]:
video_properties

{'62805_6': {'width': 480,
  'height': 600,
  'fps': 30.0,
  'frame_count': 48,
  'duration_sec': 1.6},
 '62866_3': {'width': 480,
  'height': 270,
  'fps': 30.0,
  'frame_count': 77,
  'duration_sec': 2.57},
 '62868_4': {'width': 480,
  'height': 600,
  'fps': 30.0,
  'frame_count': 73,
  'duration_sec': 2.43},
 '62876_2': {'width': 480,
  'height': 480,
  'fps': 30.0,
  'frame_count': 82,
  'duration_sec': 2.73},
 '62947_2': {'width': 480,
  'height': 480,
  'fps': 30.0,
  'frame_count': 116,
  'duration_sec': 3.87},
 '62989_7': {'width': 480,
  'height': 480,
  'fps': 30.0,
  'frame_count': 85,
  'duration_sec': 2.83},
 '62992_5': {'width': 480,
  'height': 480,
  'fps': 30.0,
  'frame_count': 160,
  'duration_sec': 5.33},
 '62993_1': {'width': 480,
  'height': 600,
  'fps': 30.0,
  'frame_count': 320,
  'duration_sec': 10.67},
 '63009_4': {'width': 480,
  'height': 270,
  'fps': 30.0,
  'frame_count': 62,
  'duration_sec': 2.07},
 '63028_3': {'width': 480,
  'height': 270,
  'fps':

In [12]:
# this detects the number of people in each video, in the frame keypoints[1] (the second person)

people_lst = []
for i in all_data.keys():
    print('No. people: ',len(all_data.get(i).get('keypoints')[1].keys()))
    people_lst.append([i, len(all_data.get(i).get('keypoints')[1].keys())])

No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  2
No. people:  1
No. people:  1
No. people:  2
No. people:  3
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  2
No. people:  2
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  3
No. people:  1
No. people:  1
No. people:  1
No. people:  2
No. people:  1
No. people:  2
No. people:  1
No. people:  1
No. people:  1
No. people:  4
No. people:  2
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  2
No. people:  2
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  2
No. people:  1
No. people:  2
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people:  1
No. people

In [13]:
# videos with no people in the second frame
for i in people_lst:
    if i[1]==0:
        print(i)




['68283_3', 0]
['69116_4', 0]


In [14]:
# How many people are in each video at most?

people_lst = []
for i in all_data.keys():
    people_visible = []
    for j in range(len(all_data.get(i).get('keypoints'))):
        if len(all_data.get(i).get('keypoints')[j])>0:
            people_visible.append(len(all_data.get(i).get('keypoints')[j].keys()))
    people_lst.append([i, max(people_visible)])
people_lst


[['62805_6', 1],
 ['62866_3', 1],
 ['62868_4', 1],
 ['62876_2', 1],
 ['62947_2', 2],
 ['62989_7', 5],
 ['62992_5', 2],
 ['62993_1', 3],
 ['63009_4', 4],
 ['63028_3', 4],
 ['63041_9', 1],
 ['63094_1', 2],
 ['63097_1', 3],
 ['63106_6', 1],
 ['63159_8', 1],
 ['63164_2', 1],
 ['63203_3', 2],
 ['63207_1', 1],
 ['63241_1', 1],
 ['63248_1', 2],
 ['63296_2', 1],
 ['63309_8', 1],
 ['63313_2', 1],
 ['63364_1', 1],
 ['63378_5', 2],
 ['63390_7', 1],
 ['63422_3', 1],
 ['63425_1', 2],
 ['63427_4', 3],
 ['63451_3', 1],
 ['63474_2', 1],
 ['63526_1', 2],
 ['63672_1', 1],
 ['63706_7', 2],
 ['63725_6', 2],
 ['63729_5', 3],
 ['63766_9', 2],
 ['63783_4', 1],
 ['63784_2', 3],
 ['63815_1', 2],
 ['63886_1', 1],
 ['63910_2', 3],
 ['63915_6', 2],
 ['63918_2', 1],
 ['63919_3', 1],
 ['63992_1', 5],
 ['63995_1', 3],
 ['64000_1', 5],
 ['64071_1', 1],
 ['64079_2', 1],
 ['64099_1', 2],
 ['64108_11', 2],
 ['64120_1', 2],
 ['64142_1', 1],
 ['64164_1', 1],
 ['64194_1', 2],
 ['64198_6', 1],
 ['64232_1', 1],
 ['64271_3', 

In [15]:
# Ressume of the max number of people per video
# Conclusion: is worth it to work with the 2 and 3 people videos.

from collections import Counter

# Get only the max number of people per video
max_people_per_video = [item[1] for item in people_lst]

# Count how many times each number appears
summary = Counter(max_people_per_video)

# Print summary sorted by number of people
for num_people in sorted(summary):
    print(f"{summary[num_people]} videos with {num_people} people at most")


261 videos with 1 people at most
122 videos with 2 people at most
53 videos with 3 people at most
13 videos with 4 people at most
9 videos with 5 people at most
1 videos with 7 people at most


In [16]:
# Get ID_video with max number of people per video, only when is greater than 1

from collections import defaultdict

# Diccionario para agrupar por número de personas máximas por video
videos_by_people_count = defaultdict(list)

# Obtener la cantidad máxima de personas por video
for video_id, video_data in all_data.items():
    max_people = 0
    for frame in video_data['keypoints']:
        people_in_frame = len(frame)
        if people_in_frame > max_people:
            max_people = people_in_frame

    if max_people > 1:
        videos_by_people_count[max_people].append(video_id)

# Mostrar resultados como: id video | N people in screen
for people_count in sorted(videos_by_people_count.keys()):
    for video_id in videos_by_people_count[people_count]:
        print(f"{video_id} | {people_count} people in screen")


62947_2 | 2 people in screen
62992_5 | 2 people in screen
63094_1 | 2 people in screen
63203_3 | 2 people in screen
63248_1 | 2 people in screen
63378_5 | 2 people in screen
63425_1 | 2 people in screen
63526_1 | 2 people in screen
63706_7 | 2 people in screen
63725_6 | 2 people in screen
63766_9 | 2 people in screen
63815_1 | 2 people in screen
63915_6 | 2 people in screen
64099_1 | 2 people in screen
64108_11 | 2 people in screen
64120_1 | 2 people in screen
64194_1 | 2 people in screen
64307_1 | 2 people in screen
64336_1 | 2 people in screen
64349_7 | 2 people in screen
64503_2 | 2 people in screen
64673_3 | 2 people in screen
64702_1 | 2 people in screen
64708_1 | 2 people in screen
64804_3 | 2 people in screen
64899_1 | 2 people in screen
64908_5 | 2 people in screen
65343_1 | 2 people in screen
65348_4 | 2 people in screen
65524_2 | 2 people in screen
65771_2 | 2 people in screen
65783_2 | 2 people in screen
65971_1 | 2 people in screen
66322_3 | 2 people in screen
66492_1 | 2 p

In [17]:
# # Copy these videos to a new folder to manually check them
# # Conclusion: nothing really crazy happening here, the videos are quite normal. The main person is visible most of the time.


# import os
# import shutil

# # Crear lista de videos con más de una persona
# multi_person_videos = []

# for video_id, video_data in all_data.items():
#     max_people = max(len(frame) for frame in video_data['keypoints'])
#     if max_people > 1:
#         multi_person_videos.append(video_id)

# # Definir carpetas
# destination_base = os.path.join(base_dir, "videos with multiple people")
# destination_class_folder = os.path.join(destination_base, sample_class)
# source_video_folder = os.path.join(base_dir, 'ohp_labeled', sample_class)

# # Crear carpetas si no existen
# os.makedirs(destination_class_folder, exist_ok=True)

# # Copiar los archivos de video
# for video_id in multi_person_videos:
#     source_file = os.path.join(source_video_folder, f"{video_id}.mp4")
#     destination_file = os.path.join(destination_class_folder, f"{video_id}.mp4")

#     if os.path.exists(source_file):
#         shutil.copy2(source_file, destination_file)
#         print(f"✅ Copied: {video_id}.mp4")
#     else:
#         print(f"⚠️ Video not found: {video_id}.mp4")

# print(f"\n🎯 Completed copying {len(multi_person_videos)} videos to {destination_class_folder}")


In [18]:
for i in people_lst:
    if i[1]>3:
        print(i)

['62989_7', 5]
['63009_4', 4]
['63028_3', 4]
['63992_1', 5]
['64000_1', 5]
['64271_3', 4]
['64482_1', 4]
['64799_1', 5]
['64893_3', 5]
['64922_5', 4]
['66242_6', 4]
['66251_9', 5]
['67066_4', 5]
['67451_2', 5]
['68880_3', 4]
['69055_3', 4]
['69254_2', 4]
['69780_6', 5]
['70924_4', 4]
['71960_2', 4]
['74030_6', 4]
['76981_4', 4]
['77575_1', 7]


In [19]:
# def get_bounding_box(keypoints, threshold=0.0):
#     """
#     keypoints: list of (x, y, confidence) or (x, y)
#     """
#     valid_points = []
#     for kp in keypoints:
#         if len(kp) == 3:
#             x, y, conf = kp
#             if conf >= threshold:
#                 valid_points.append((x, y))
#         elif len(kp) == 2:
#             x, y = kp
#             valid_points.append((x, y))

#     if not valid_points:
#         return None

#     xs, ys = zip(*valid_points)
#     return min(xs), min(ys), max(xs), max(ys)

In [20]:
# def bbox_area(bbox):
#     x_min, y_min, x_max, y_max = bbox
#     return (x_max - x_min) * (y_max - y_min)

In [21]:
#all_data.get('80557_5').get('keypoints')

In [22]:
#video_properties.get('80557_5')

In [23]:
len(all_data.keys())

459

In [24]:
# # Dont use this if you want all the people in the video.
# # This bbg extracts "the main person" from each video based on the largest bounding box area of keypoints.
# # This "main person" is defined as the person with the largest bounding box in the **first frame** of each video.
# # Should redefine that 


# main_person = None
# counter = 0
# main_person_keypoints = {}

# if not extract_main_person:
#     exit()

# for video in all_data.keys():
#     main_person = None
#     largest_area = 0
#     for frame in all_data.get(video).get('keypoints'):
#         if len(frame.keys())>0:
#             for (person, person_keypoints) in frame.items():  # each is a list of keypoints
#                 bbox = get_bounding_box(person_keypoints, threshold=0.2)  # optional threshold
#                 if bbox:
#                     area = bbox_area(bbox)
#                     if area > largest_area:
#                         largest_area = area
#                         main_person = {
#                             "bbox": bbox,
#                             "keypoints": person_keypoints,
#                             "area": area,
#                             "person_id": person
#                         }
#             counter+=1
#             if main_person:
#                 print("Main person bounding box:", main_person["bbox"], video, counter)
#                 print(main_person['area'])
#                 print(main_person['person_id'])
#             break
#     main_persons_frames = []
#     for frame in all_data.get(video).get('keypoints'):
#         if frame.get(main_person['person_id']):
#             main_persons_frames.append(frame.get(main_person['person_id'])[:17])
        
#     main_person_keypoints[video] = {main_person['person_id']: main_persons_frames}


In [25]:
import itertools

all_keypoints= {}
for video in all_data.keys():
    persons_frames = {}
    all_keypoints[video] = []
     
    people_lst = list([list(all_data.get(video).get('keypoints')[i].keys()) for i in range(len(all_data.get(video).get('keypoints')))])
    people_set = list(set(itertools.chain.from_iterable(people_lst)))
    for person in people_set:
            persons_frames[person] = []
    
    for frame in all_data.get(video).get('keypoints'):
        for person in frame.keys():
            persons_frames[person].append(frame.get(person)[:17])
            
    all_keypoints[video].append(persons_frames)


In [26]:
# len(all_keypoints['62794_6'][0]['44'])

In [27]:
#main_person_keypoints.get('80756_1').get('1860')[0]

In [28]:
# if extract_main_person:
#     keypoints = main_person_keypoints.keys()
# else:
#     pass

In [29]:
# pip install scikit-learn

In [30]:
import random
from sklearn.model_selection import train_test_split

video_ids = list(all_keypoints.keys())
random.seed(42)

train_val, test = train_test_split(video_ids, test_size=0.10, random_state=42)

val_size = 0.1111
train, val = train_test_split(train_val, test_size=val_size, random_state=42)

split = {
    'train': train,
    'val': val,
    'test': test
}

In [31]:
print(len(train), len(test), len(val))

367 46 46


In [32]:
import numpy as np

coords = {}
confidences = {}

# Iteramos sobre todos los videos en all_keypoints
for video_id in all_keypoints.keys():
    persons_data = all_keypoints[video_id][0]  # dict: person_id -> list of frames

    person_ids = list(persons_data.keys())
    num_persons = len(person_ids)
    num_frames = max(len(persons_data[pid]) for pid in person_ids)
    num_keypoints = len(persons_data[person_ids[0]][0])  # assumed 17 keypoints

    # Inicializamos arrays vacíos
    keypoint_array = np.zeros((num_persons, num_frames, num_keypoints, 2), dtype='float32')
    score_array = np.zeros((num_persons, num_frames, num_keypoints), dtype='float32')


    for m, pid in enumerate(person_ids):
        frames = persons_data[pid]
        for t, frame in enumerate(frames):
            for v, kp in enumerate(frame):
                keypoint_array[m, t, v] = kp[:2]
                score_array[m, t, v] = kp[2] if len(kp) > 2 else 0.0

    coords[video_id] = keypoint_array
    confidences[video_id] = score_array


In [33]:
final_dict = {}
final_dict['split'] = {'train': train, 'test': test, 'val': val}
final_dict['annotations'] = []

for video_id in all_keypoints.keys():
    final_dict['annotations'].append({
        'frame_dir': video_id,
        'total_frames': video_properties[video_id]['frame_count'],
        'img_shape': (video_properties[video_id]['height'], video_properties[video_id]['width']),
        'original_shape': (video_properties[video_id]['height'], video_properties[video_id]['width']),
        'label': 0,
        'keypoint': coords[video_id],  # shape [M, T, V, C]
        'keypoint_score': confidences[video_id]  # shape [M, T, V]
    })


In [34]:
final_dict['annotations'][0]

{'frame_dir': '62805_6',
 'total_frames': 48,
 'img_shape': (600, 480),
 'original_shape': (600, 480),
 'label': 0,
 'keypoint': array([[[[148.6352  , 170.44833 ],
          [142.7023  , 163.54913 ],
          [145.70682 , 161.94452 ],
          ...,
          [511.09167 , 235.31567 ],
          [570.91364 , 109.866425],
          [603.3789  , 223.05258 ]],
 
         [[149.73306 , 169.0805  ],
          [177.24146 , 105.07037 ],
          [147.01593 , 161.0906  ],
          ...,
          [510.97034 , 235.97339 ],
          [569.745   , 109.49628 ],
          [603.8702  , 222.57098 ]],
 
         [[150.8147  , 165.46503 ],
          [145.33044 , 159.0509  ],
          [148.50717 , 157.03455 ],
          ...,
          [509.3875  , 234.99701 ],
          [569.5072  , 109.370544],
          [603.33276 , 222.36609 ]],
 
         ...,
 
         [[149.0133  , 174.24326 ],
          [139.87799 , 167.23444 ],
          [142.00195 , 164.61475 ],
          ...,
          [518.36304 , 224.8772

In [ ]:
import pickle

with open(os.path.join(f'{sample_class}.pkl'), 'wb') as handle:
    pickle.dump(final_dict, handle, protocol=pickle.HIGHEST_PROTOCOL)


In [36]:
from joblib import load

obj = load("correct.pkl")
print(type(obj))

<class 'dict'>


In [37]:
obj.get('split')

{'train': ['72030_1',
  '67154_1',
  '79696_10',
  '77441_1',
  '65041_4',
  '73476_5',
  '72034_2',
  '72320_2',
  '71698_2',
  '73688_1',
  '73389_2',
  '64620_1',
  '75938_4',
  '77124_1',
  '67412_4',
  '78002_2',
  '65117_5',
  '62959_3',
  '76587_2',
  '71165_4',
  '79155_1',
  '68603_5',
  '66888_1',
  '67692_8',
  '78928_4',
  '76310_1',
  '70314_2',
  '75901_2',
  '77577_14',
  '67816_1',
  '74258_1',
  '73117_3',
  '74250_3',
  '66449_11',
  '66014_8',
  '75486_1',
  '77722_2',
  '74043_2',
  '78783_3',
  '71677_1',
  '65867_4',
  '80056_3',
  '75215_5',
  '73012_3',
  '71195_9',
  '65301_2',
  '80350_2',
  '70105_1',
  '74235_2',
  '73147_1',
  '64354_1',
  '65555_6',
  '67588_2',
  '74856_1',
  '69107_1',
  '76760_4',
  '63206_1',
  '75937_5',
  '71357_3',
  '71938_1',
  '72126_5',
  '79144_4',
  '67396_2',
  '68648_1',
  '80677_3',
  '72284_1',
  '65290_9',
  '79572_1',
  '71350_5',
  '73979_3',
  '66146_3',
  '68897_4',
  '63777_2',
  '69478_1',
  '76518_9',
  '77175_1',


In [38]:
obj.get('annotations')

[{'frame_dir': '62794_6',
  'total_frames': 71,
  'img_shape': (270, 480),
  'original_shape': (270, 480),
  'label': 0,
  'keypoint': array([[[[118.3231  , 189.48622 ],
           [116.60971 , 190.8407  ],
           [116.63011 , 187.42722 ],
           ...,
           [181.87386 , 181.7889  ],
           [203.20834 , 195.27698 ],
           [203.4859  , 180.34311 ]],
  
          [[116.45407 , 189.72186 ],
           [114.90763 , 191.09082 ],
           [114.85316 , 187.5405  ],
           ...,
           [181.2757  , 181.68169 ],
           [202.44328 , 194.82301 ],
           [203.1583  , 180.15706 ]],
  
          [[117.53462 , 189.88078 ],
           [115.9615  , 191.35829 ],
           [115.88116 , 187.8663  ],
           ...,
           [181.69775 , 181.69965 ],
           [203.11244 , 194.73088 ],
           [203.77283 , 180.10722 ]],
  
          ...,
  
          [[117.47156 , 189.17133 ],
           [116.101776, 189.87744 ],
           [115.97197 , 187.30997 ],
           .